# CyberVision Gen — Modèle A : WGAN-GP conditionnel

## 1. Présentation du projet et objectifs du modèle A

CyberVision Gen étudie la synthèse conditionnelle de matrices de caractéristiques d'intrusion issues de CIC-IDS2017. Le modèle A est un **WGAN-GP conditionnel tabulaire** : il génère 63 caractéristiques selon l'une des 11 classes, puis les vecteurs peuvent être projetés dans une matrice 8×8 avec une case de remplissage.

Ce notebook Colab couvre la configuration, l'entraînement, la sélection du checkpoint, la génération, les contrôles de qualité et l'export. Le jeu de test n'est pas chargé ici.

> **Traçabilité des résultats.** Les chiffres ci-dessous sont les résultats expérimentaux rapportés par le run Colab antérieur. Ils ne sont pas présentés comme reproduits pendant la préparation de cette version GitHub.

- entraînement conditionnel WGAN-GP sur 100 époques ;
- meilleur checkpoint à l'époque 50, pour une distance de validation minimale de 0,071821 ;
- 23 000 exemples générés sur 11 classes : 5 000 pour Bot, PortScan et DoS Slowhttptest, puis 1 000 pour chacune des huit autres classes ;
- aucun doublon exact ni quasi-doublon détecté parmi les 15 000 exemples prioritaires selon les contrôles réalisés ;
- 100 % des lignes synthétiques violent au moins une contrainte fiable, avant comme après post-traitement.

Les limites restent majeures : plausibilité physique insuffisante, audit temporel des données non finalisé et utilité pour la classification non encore démontrée.

### Guide de lecture

Les sorties du notebook ont été retirées pour GitHub. Les cellules enregistrent néanmoins les checkpoints, Parquet, rapports CSV et figures dans le dossier persistant configuré sur Google Drive.

Les contrôles techniques attendus incluent les formes `(B, 63)` et `(B, 1, 8, 8)`, la finitude de la gradient penalty, ainsi que la présence des 11 classes. Les métriques affichées lors d'une future exécution ne doivent pas être confondues avec les résultats expérimentaux rapportés ci-dessus.

## 2. Configuration de Google Colab et des dépendances

Installation des dépendances légères, initialisation des graines aléatoires et sélection du périphérique. PyTorch fourni par Colab est conservé afin de ne pas remplacer sa version CUDA.

In [ ]:
%pip install -q pyarrow pandas pyyaml matplotlib seaborn pytest
import os, sys, json, random, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch={torch.__version__} | device={DEVICE}')

## 3. Connexion Google Drive et configuration des chemins

Les cellules suivantes importent l'archive du dépôt dans Colab, identifient sa racine, montent Google Drive et définissent les chemins persistants. Cette séquence reflète le run Colab de référence et reste dans son ordre d'origine.

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from pathlib import Path
import zipfile

zip_path = Path(
    "/content/cybervision-gen-personne2-wgan-gp.zip"
)

assert zip_path.is_file(), "ZIP introuvable dans Colab"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content")

print("✅ ZIP extrait avec succès")

In [ ]:
import os
from pathlib import Path

projects = [
    p for p in Path("/content").iterdir()
    if p.is_dir()
    and (p / "cybervision_gan").is_dir()
    and (p / "cybervision_data").is_dir()
]

assert len(projects) == 1, (
    f"Nombre de projets trouvés : {len(projects)}"
)

PROJECT_ROOT = projects[0]

os.environ["CYBERVISION_PROJECT_ROOT"] = str(PROJECT_ROOT)

print("✅ Projet trouvé :", PROJECT_ROOT)
print("✅ Dossier GAN :", (PROJECT_ROOT / "cybervision_gan").exists())

La cellule précédente affiche la version de PyTorch et le périphérique retenu (`cpu` ou `cuda`).

### Montage de Drive et chemins persistants

Le dépôt, les données privées et les sorties sont localisés explicitement. Les checkpoints et livrables restent sur Drive ; aucun chemin de sauvegarde du run existant n'est modifié.

In [ ]:

import os
import sys
from pathlib import Path

# 1. Monter Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Hors Colab : montage Drive ignoré.")

# 2. Rechercher le projet extrait depuis le ZIP
projects = [
    p for p in Path("/content").iterdir()
    if p.is_dir()
    and (p / "cybervision_gan").is_dir()
    and (p / "cybervision_data").is_dir()
]

if len(projects) == 1:
    PROJECT_ROOT = projects[0]
elif len(projects) > 1:
    raise RuntimeError(
        f"Plusieurs projets trouvés : {projects}"
    )
else:
    raise FileNotFoundError(
        "Projet introuvable dans /content. "
        "Importez et extrayez d'abord le ZIP GitHub."
    )

os.environ["CYBERVISION_PROJECT_ROOT"] = str(PROJECT_ROOT)
print("✅ Projet trouvé :", PROJECT_ROOT)

# 3. Dossier des artefacts
ARTIFACTS_DIR = (
    PROJECT_ROOT / "cybervision_data" / "artifacts"
)

assert ARTIFACTS_DIR.is_dir(), "Artefacts introuvables"

# 4. Rechercher les Parquet sur Google Drive
drive_root = Path("/content/drive/MyDrive")
required_names = [
    "train_scaled.parquet",
    "val_scaled.parquet"
]

configured_dir = os.environ.get(
    "CYBERVISION_PROCESSED_DIR"
)

if configured_dir:
    candidates = [Path(configured_dir)]
else:
    candidates = [drive_root]

found = {}

for filename in required_names:
    matches = []

    for directory in candidates:
        if directory.is_dir():
            matches.extend(directory.rglob(filename))

    if len(matches) == 1:
        found[filename] = matches[0]
        print(f"✅ {filename} : {matches[0]}")
    elif len(matches) == 0:
        print(f"❌ {filename} introuvable")
    else:
        raise RuntimeError(
            f"Plusieurs copies de {filename} : {matches}"
        )

if len(found) != 2:
    raise FileNotFoundError(
        "Les fichiers train_scaled.parquet et "
        "val_scaled.parquet doivent être accessibles "
        "depuis Google Drive."
    )

# 5. Copier les données dans le projet si nécessaire
import shutil

PROCESSED_DIR = (
    PROJECT_ROOT / "cybervision_data" / "data" / "processed"
)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

for filename, source in found.items():
    target = PROCESSED_DIR / filename

    if not target.exists():
        shutil.copy2(source, target)

    print(f"✅ Fichier disponible : {target}")

# 6. Dossier persistant pour les résultats GAN
OUTPUT_DIR = (
    drive_root / "CyberVision_GAN_Results" / "model_a"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 7. Ajouter les modules Python au chemin d'import
for path in [
    PROJECT_ROOT,
    PROJECT_ROOT / "cybervision_data"
]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

# 8. Afficher la configuration finale
print("\n========== CONFIGURATION ==========")
print("Dépôt :", PROJECT_ROOT)
print("Données :", PROCESSED_DIR)
print("Artefacts :", ARTIFACTS_DIR)
print("Résultats :", OUTPUT_DIR)
print("===================================")


## 4. Chargement et vérification de CIC-IDS2017

Les partitions d'entraînement et de validation, l'ordre des 63 caractéristiques et l'encodage des 11 classes proviennent des artefacts existants. Les schémas, plages, valeurs finies et labels sont contrôlés avant utilisation.

## 5. Préparation du Dataset et du DataLoader PyTorch

La même cellule construit les tableaux NumPy et les DataLoader équilibrés à partir de la configuration existante. Ce regroupement est conservé afin de ne pas toucher au code exécutable ni à son ordre.

In [ ]:
from cybervision_gan.training.train import create_dataloader, set_seed
from cvgen_data.matrix import MatrixLayout

with open(PROJECT_ROOT / 'cybervision_gan/configs/model_a.yaml', encoding='utf-8') as handle:
    cfg = yaml.safe_load(handle)
with open(ARTIFACTS_DIR / 'matrix_layout.json', encoding='utf-8') as handle:
    layout_json = json.load(handle)
with open(ARTIFACTS_DIR / 'label_encoding.json', encoding='utf-8') as handle:
    label_to_index = json.load(handle)
features = layout_json['features']
assert len(features) == layout_json['n_features'] == cfg['model']['n_features'] == 63
assert len(label_to_index) == cfg['model']['n_classes'] == 11
assert sorted(label_to_index.values()) == list(range(11))
assert np.asarray(layout_json['mask']).sum() == 63
layout = MatrixLayout.from_json(str(ARTIFACTS_DIR / 'matrix_layout.json'))

train_df = pd.read_parquet(PROCESSED_DIR / 'train_scaled.parquet')
val_df = pd.read_parquet(PROCESSED_DIR / 'val_scaled.parquet')
for split_name, frame in [('train', train_df), ('validation', val_df)]:
    missing_columns = set(features + ['label']) - set(frame.columns)
    if missing_columns: raise KeyError(f'{split_name}: colonnes absentes {sorted(missing_columns)}')
    unknown = set(frame['label'].dropna().unique()) - set(label_to_index)
    if unknown: raise ValueError(f'{split_name}: labels inconnus {sorted(unknown)}')
    values = frame[features].to_numpy(dtype=np.float32)
    if not np.isfinite(values).all(): raise ValueError(f'{split_name}: NaN ou infini détecté')
    print(split_name, frame.shape, 'plage=', (float(values.min()), float(values.max())))

X_train = train_df[features].to_numpy(dtype=np.float32)
X_val = val_df[features].to_numpy(dtype=np.float32)
y_train = train_df['label'].map(label_to_index).to_numpy(dtype=np.int64)
y_val = val_df['label'].map(label_to_index).to_numpy(dtype=np.int64)
if X_train.min() < -1.0001 or X_train.max() > 1.0001:
    raise ValueError("Le train n'est pas dans la plage contractuelle [-1,1].")
if X_val.min() < -1.0001 or X_val.max() > 1.0001:
    print('Attention : validation hors plage train, autorisé par le pipeline (non-écrêté).')
set_seed(cfg['project']['seed'])
train_loader = create_dataloader(
    X_train, y_train, cfg['training']['batch_size'], balanced=True, seed=SEED,
    num_workers=cfg['training']['num_workers'], drop_last=True)
val_loader = create_dataloader(
    X_val, y_val, cfg['training']['batch_size'], balanced=True, seed=SEED + 1,
    num_workers=cfg['training']['num_workers'], drop_last=True)
print('Effectifs train :', train_df['label'].value_counts().to_dict())

Les formes, plages numériques et effectifs par classe sont affichés lors de l'exécution. Aucune valeur de sortie n'est intégrée dans cette version nettoyée.

## 6. Architecture du générateur conditionnel

Le générateur concatène un bruit de dimension 128 et un embedding de classe de dimension 32. Son MLP utilise les dimensions cachées `[256, 512, 512]`, puis produit 63 valeurs avec une activation `tanh`. Ces paramètres sont lus depuis `model_a.yaml`.

In [ ]:
from cybervision_gan.models import Generator, Critic
def build_models(config):
    model = config['model']
    generator = Generator(model['latent_dim'], model['n_classes'], model['n_features'],
                          model['embedding_dim'], model['generator_hidden_dims'])
    critic = Critic(model['n_classes'], model['n_features'], model['embedding_dim'],
                    model['critic_hidden_dims'])
    return generator, critic
generator, critic = build_models(cfg)
z = torch.randn(4, cfg['model']['latent_dim'])
demo_labels = torch.tensor([label_to_index[x] for x in ['PortScan','Bot','DoS Slowhttptest','BENIGN']])
demo = generator(z, demo_labels)
print(generator)
print('Sortie G :', tuple(demo.shape), float(demo.min()), float(demo.max()))

## 7. Architecture du critique conditionnel

Le critique concatène les 63 caractéristiques et l'embedding de classe. Son MLP `[512, 512, 256]` retourne un score réel sans sigmoid ni BatchNorm, conformément à l'objectif Wasserstein.

In [ ]:
scores = critic(demo, demo_labels)
print(critic)
print('Scores C :', tuple(scores.shape), scores.detach().numpy())

## 8. Fonction de perte WGAN-GP et gradient penalty

La pénalité de gradient interpole des exemples réels et synthétiques de même classe et pénalise l'écart de la norme du gradient à 1. Le calcul existant et son poids configuré à 10 ne sont pas modifiés.

In [ ]:
from cybervision_gan.training.gradient_penalty import gradient_penalty
real_batch, label_batch = next(iter(train_loader))
small_n = min(16, len(real_batch))
fake_batch = generator(torch.randn(small_n, cfg['model']['latent_dim']), label_batch[:small_n])
gp = gradient_penalty(critic, real_batch[:small_n], fake_batch, label_batch[:small_n])
gp.backward()
assert torch.isfinite(gp) and any(p.grad is not None for p in critic.parameters())
generator.zero_grad(set_to_none=True); critic.zero_grad(set_to_none=True)
print('Gradient penalty finie :', float(gp.detach()))

## 9. Smoke test technique

Le smoke test utilise des réseaux neufs et un dossier de checkpoints séparé. Il vérifie cinq batches, soit un cycle `n_critic`, sans contaminer le modèle complet.

In [ ]:
from cybervision_gan.training.train import train_wgan_gp
smoke_g, smoke_c = build_models(cfg)
smoke_history = train_wgan_gp(
    smoke_g, smoke_c, train_loader, val_loader, cfg, OUTPUT_DIR / 'smoke_checkpoints',
    smoke_test=True)
assert len(smoke_history) == 1
print('Smoke test réussi. Ces métriques ne constituent pas un résultat expérimental.')

## 10. Entraînement complet et sauvegarde des checkpoints

L'entraînement complet utilise les 100 époques, `n_critic=5`, Adam et les hyperparamètres du fichier de configuration. Le garde-fou `RUN_FULL_TRAINING=False` évite tout relancement accidentel ; la reprise éventuelle s'appuie sur `latest.pt` et les sauvegardes restent sur Drive.

In [ ]:
# Passez à True uniquement après le smoke test. Les checkpoints restent sur Drive.
RUN_FULL_TRAINING = True
RESUME = True
generator, critic = build_models(cfg)
full_checkpoint_dir = OUTPUT_DIR / 'checkpoints'
latest = full_checkpoint_dir / 'latest.pt'
resume_from = latest if (RUN_FULL_TRAINING and RESUME and latest.exists()) else None
if RUN_FULL_TRAINING:
    history = train_wgan_gp(generator, critic, train_loader, val_loader, cfg,
                                full_checkpoint_dir, resume_from=resume_from)
else:
    history = smoke_history
    print('Entraînement complet désactivé. Fixez RUN_FULL_TRAINING=True pour le lancer/reprendre.')

## 11. Sélection et rechargement du meilleur checkpoint

Les cellules inspectent les métriques persistées, identifient le minimum de distance de validation et rechargent `best.pt`. Le run rapporté a sélectionné l'époque 50 avec une distance minimale de 0,071821.

In [ ]:
from pathlib import Path

checkpoint_dir = OUTPUT_DIR / "checkpoints"

print("Dossier :", checkpoint_dir)
print("Existe :", checkpoint_dir.exists())

if checkpoint_dir.exists():
    for file in sorted(checkpoint_dir.iterdir()):
        if file.is_file():
            print(
                file.name,
                "-",
                round(file.stat().st_size / (1024**2), 2),
                "Mo"
            )

In [ ]:
import torch

checkpoint_path = OUTPUT_DIR / "checkpoints" / "best.pt"

# Charger le checkpoint sur CPU pour inspecter ses métadonnées
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Clés disponibles dans le checkpoint :")
print(list(checkpoint.keys()))

print("\nInformations sur l'époque :")
for key in ["epoch", "best_epoch", "best_val_distance", "best_metric"]:
    if key in checkpoint:
        print(f"{key} : {checkpoint[key]}")

In [ ]:
import json

metrics_path = OUTPUT_DIR / "checkpoints" / "metrics.json"

with open(metrics_path, "r") as f:
    metrics = json.load(f)

print("Type de structure :", type(metrics).__name__)

if isinstance(metrics, list):
    best = min(
        metrics,
        key=lambda row: row["val_distribution_distance"]
    )

    print("\nMeilleure époque :", best["epoch"])
    print(
        "Meilleure distance :",
        best["val_distribution_distance"]
    )
else:
    print("Clés disponibles :", list(metrics.keys()))

In [ ]:
best_checkpoint = torch.load(
    OUTPUT_DIR / "checkpoints" / "best.pt",
    map_location="cpu",
    weights_only=False
)

generator, critic = build_models(cfg)

generator.load_state_dict(best_checkpoint["generator"])
generator.eval()

print("Générateur chargé avec succès")
print("Époque :", best_checkpoint["epoch"])
print("Device :", next(generator.parameters()).device)

In [ ]:
import json

# Charger l'encodage des classes créé par la personne 1
labels_path = ARTIFACTS_DIR / "label_encoding.json"

with open(labels_path, "r", encoding="utf-8") as f:
    label_to_id = json.load(f)

# Les trois classes prioritaires
target_classes = [
    "PortScan",
    "Bot",
    "DoS Slowhttptest"
]

print("Nombre total de classes :", len(label_to_id))

for name in target_classes:
    print(name, "->", label_to_id[name])

In [ ]:
import torch

generator.eval()

print("Époque utilisée :", best_checkpoint["epoch"])
print("Device :", next(generator.parameters()).device)
print("Classes cibles :", target_classes)

Les métriques d'entraînement (`critic_loss`, `generator_loss`, `gradient_penalty` et `val_distribution_distance`) sont conservées dans les fichiers persistants. Les sorties incorporées au notebook ont été supprimées.

## 12. Génération conditionnelle des 11 classes

Le meilleur checkpoint complet est privilégié. La génération produit 5 000 vecteurs pour chacune des trois classes prioritaires et 1 000 pour chacune des huit autres classes, soit 23 000 exemples rapportés.

## 13. Transformation des vecteurs en matrices 8×8

La cellule applique également le layout existant aux classes prioritaires et vérifie la forme `(B, 1, 8, 8)` ainsi que la case de remplissage `(7, 7)`. Les deux opérations restent réunies pour préserver intégralement le code.

In [ ]:
from cybervision_gan.evaluation.generate import generate_conditioned, vectors_to_matrices
from cybervision_gan.training.train import load_checkpoint
import json
import numpy as np

# Sécuriser les variables nécessaires
if "label_to_index" not in globals():
    with open(ARTIFACTS_DIR / "label_encoding.json", "r", encoding="utf-8") as f:
        label_to_index = json.load(f)

if "layout_json" not in globals():
    with open(ARTIFACTS_DIR / "matrix_layout.json", "r", encoding="utf-8") as f:
        layout_json = json.load(f)

priority = cfg["project"]["priority_classes"]

best_full = OUTPUT_DIR / "checkpoints" / "best.pt"
best_smoke = OUTPUT_DIR / "smoke_checkpoints" / "best.pt"

selected_checkpoint = best_full if best_full.exists() else best_smoke

if not selected_checkpoint.exists():
    raise FileNotFoundError(
        "Aucun checkpoint trouvé : exécutez d'abord l'entraînement."
    )

generator, critic = build_models(cfg)
state = load_checkpoint(selected_checkpoint, generator, critic, map_location=DEVICE)
generator.eval()

if selected_checkpoint == best_smoke:
    print("ATTENTION : checkpoint smoke utilisé ; sorties techniques uniquement.")
else:
    print("Checkpoint utilisé :", selected_checkpoint)
    print("Époque chargée :", state.get("epoch", "inconnue"))

samples_per_class = {
    name: (5000 if name in priority else 1000)
    for name in label_to_index
}

generated_by_class = {
    name: generate_conditioned(
        generator,
        index,
        samples_per_class[name],
        cfg["model"]["latent_dim"],
        DEVICE,
        cfg["generation"]["batch_size"],
    )
    for name, index in label_to_index.items()
}

for name, values in generated_by_class.items():
    assert values.shape == (samples_per_class[name], 63), (
        f"Forme incorrecte pour {name}: {values.shape}"
    )

priority_matrices = {
    name: vectors_to_matrices(generated_by_class[name], layout_json)
    for name in priority
}

assert all(
    m.shape[1:] == (1, 8, 8) and np.all(m[:, 0, 7, 7] == 0)
    for m in priority_matrices.values()
), "Erreur dans la conversion en matrices 8×8"

print("Génération réussie.")
print({name: values.shape for name, values in generated_by_class.items()})

## 14. Visualisations et analyses statistiques

Les distributions marginales réelles/synthétiques sont comparées pour les classes prioritaires, puis les distances de Wasserstein sont enregistrées dans le dossier d'analyse sur Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

priority_classes = [
    "PortScan",
    "Bot",
    "DoS Slowhttptest"
]

selected_features = [
    "flow_duration",
    "flow_bytes_s",
    "flow_packets_s"
]

fig, axes = plt.subplots(3, 3, figsize=(15, 11))

for row, class_name in enumerate(priority_classes):
    real = train_df[train_df["label"] == class_name]
    fake = generated_by_class[class_name]

    for col, feature in enumerate(selected_features):
        idx = features.index(feature)
        ax = axes[row, col]

        ax.hist(
            real[feature].to_numpy(),
            bins=40,
            density=True,
            histtype="step",
            linewidth=1.8,
            label="Réel"
        )

        ax.hist(
            fake[:, idx],
            bins=40,
            density=True,
            histtype="step",
            linewidth=1.8,
            label="Synthétique"
        )

        ax.set_title(f"{class_name} — {feature}")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.2)

plt.tight_layout()

figures_dir = OUTPUT_DIR / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

plt.savefig(
    figures_dir / "03_real_vs_synthetic.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance

priority_classes = [
    "PortScan",
    "Bot",
    "DoS Slowhttptest"
]

results = []

for class_name in priority_classes:
    real = train_df.loc[
        train_df["label"] == class_name, features
    ].to_numpy(dtype=np.float32)

    fake = generated_by_class[class_name]

    distances = []

    for i, feature in enumerate(features):
        d = wasserstein_distance(real[:, i], fake[:, i])
        distances.append(d)

        results.append({
            "class": class_name,
            "feature": feature,
            "wasserstein_distance": d
        })

    print(
        f"{class_name}: "
        f"distance moyenne = {np.mean(distances):.6f}, "
        f"distance maximale = {np.max(distances):.6f}"
    )

distances_df = pd.DataFrame(results)

# Enregistrement sur Google Drive
analysis_dir = OUTPUT_DIR / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)

distances_df.to_csv(
    analysis_dir / "marginal_wasserstein_distances.csv",
    index=False
)

print("\nRésultats enregistrés dans :", analysis_dir)

## 15. Mesure des contraintes physiques

Les données réelles brutes et les deux variantes synthétiques sont confrontées aux contraintes fiables du schéma. Le résultat rapporté est un taux de violation d'au moins une contrainte de 100 % pour les lignes synthétiques, avant et après post-traitement.

In [ ]:
print("Train :", PROCESSED_DIR / "train_scaled.parquet")
print("Validation :", PROCESSED_DIR / "val_scaled.parquet")
print("Dossier :", PROCESSED_DIR)

In [ ]:
from pathlib import Path
import shutil

source = Path(
    "/content/drive/MyDrive/CyberVision_Data/train_raw.parquet"
)

destination = PROCESSED_DIR / "train_raw.parquet"

assert source.is_file(), "Fichier introuvable sur Drive"

shutil.copy2(source, destination)

print("Fichier copié :", destination)
print("Copie réussie :", destination.is_file())

In [ ]:
import json
import pandas as pd

from cvgen_data.constraints import (
    row_violation_rate,
    violation_report
)

# Charger le schéma contenant les contraintes
with open(ARTIFACTS_DIR / "feature_schema.json", encoding="utf-8") as f:
    schema = json.load(f)

# Charger les données synthétiques déjà exportées
export_dir = OUTPUT_DIR / "generated"

synthetic_raw = pd.read_parquet(
    export_dir / "synthetic_raw_inverse.parquet"
)

synthetic_post = pd.read_parquet(
    export_dir / "synthetic_postprocessed.parquet"
)

# Charger les données réelles avant normalisation
real_raw_path = PROCESSED_DIR / "train_raw.parquet"

if not real_raw_path.exists():
    print("ATTENTION : train_raw.parquet est manquant.")
    print("Il faut le récupérer depuis le Drive de la personne 1.")
else:
    real_raw = pd.read_parquet(real_raw_path)

    # Conserver uniquement les caractéristiques nécessaires
    real_features = real_raw[features]
    synthetic_features = synthetic_raw[features]
    post_features = synthetic_post[features]

    results = {
        "Train réel": row_violation_rate(
            real_features, schema, only_reliable=True
        ),
        "Synthétique brut A": row_violation_rate(
            synthetic_features, schema, only_reliable=True
        ),
        "Synthétique post-traité": row_violation_rate(
            post_features, schema, only_reliable=True
        )
    }

    print("\n=== TAUX DE VIOLATIONS ===")

    for name, rate in results.items():
        print(f"{name}: {rate * 100:.2f}%")

    results_df = pd.DataFrame(
        [{"dataset": name, "violation_rate": rate}
         for name, rate in results.items()]
    )

    analysis_dir = OUTPUT_DIR / "analysis"
    analysis_dir.mkdir(parents=True, exist_ok=True)

    results_df.to_csv(
        analysis_dir / "constraint_violation_rates.csv",
        index=False
    )

    print("\nRésultats sauvegardés.")

In [ ]:
from cvgen_data.constraints import violation_report

# Rapports détaillés
report_real = violation_report(
    real_raw[features],
    schema
)

report_synthetic = violation_report(
    synthetic_raw[features],
    schema
)

report_post = violation_report(
    synthetic_post[features],
    schema
)

print("=== COLONNES DU RAPPORT ===")
print(report_synthetic.columns.tolist())

print("\n=== PREMIÈRES VIOLATIONS SYNTHÉTIQUES ===")
display(report_synthetic.head(10))

print("\n=== PREMIÈRES VIOLATIONS POST-TRAITÉES ===")
display(report_post.head(10))

In [ ]:

from cvgen_data.constraints import row_violation_rate
import pandas as pd

results_by_class = []

for class_name in [
    "PortScan",
    "Bot",
    "DoS Slowhttptest"
]:
    real_class = real_raw.loc[
        real_raw["label"] == class_name, features
    ]

    raw_class = synthetic_raw.loc[
        synthetic_raw["label"] == class_name, features
    ]

    post_class = synthetic_post.loc[
        synthetic_post["label"] == class_name, features
    ]

    for dataset_name, data in [
        ("Train réel", real_class),
        ("Synthétique brut A", raw_class),
        ("Synthétique post-traité", post_class)
    ]:
        if data.empty:
            raise ValueError(
                f"Aucune donnée pour {class_name} — {dataset_name}"
            )

        rate = row_violation_rate(
            data, schema, only_reliable=True
        )

        results_by_class.append({
            "classe": class_name,
            "dataset": dataset_name,
            "n_samples": len(data),
            "violation_rate_pct": rate * 100
        })

violations_by_class = pd.DataFrame(results_by_class)

display(
    violations_by_class.pivot(
        index="classe",
        columns="dataset",
        values="violation_rate_pct"
    ).round(2)
)

analysis_dir = OUTPUT_DIR / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)

violations_by_class.to_csv(
    analysis_dir / "violations_by_class.csv",
    index=False
)

print("Résultats sauvegardés.")


## 16. Contrôle des doublons et quasi-doublons

Les contrôles combinent des empreintes SHA-256 pour les doublons exacts et une recherche du plus proche voisin avec les seuils par classe. Aucun doublon exact ou quasi-doublon n'a été détecté parmi les 15 000 exemples prioritaires lors du run rapporté.

In [ ]:

import numpy as np
import pandas as pd
import hashlib

priority_classes = [
    "PortScan",
    "Bot",
    "DoS Slowhttptest"
]

results_duplicates = []

def hash_rows(array):
    """Empreintes SHA-256 des vecteurs float32."""
    array = np.ascontiguousarray(
        array, dtype=np.float32
    )

    return [
        hashlib.sha256(row.tobytes()).digest()
        for row in array
    ]

for class_name in priority_classes:

    real = train_df.loc[
        train_df["label"] == class_name,
        features
    ].to_numpy(dtype=np.float32)

    fake = np.asarray(
        generated_by_class[class_name],
        dtype=np.float32
    )

    real_hashes = set(hash_rows(real))
    fake_hashes = hash_rows(fake)

    duplicate_count = sum(
        h in real_hashes for h in fake_hashes
    )

    rate = 100 * duplicate_count / len(fake_hashes)

    results_duplicates.append({
        "classe": class_name,
        "n_generated": len(fake_hashes),
        "duplicates_exact": duplicate_count,
        "duplicate_rate_pct": rate
    })

duplicates_df = pd.DataFrame(results_duplicates)

display(duplicates_df)

analysis_dir = OUTPUT_DIR / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)

duplicates_df.to_csv(
    analysis_dir / "exact_duplicates_by_class.csv",
    index=False
)

print("Analyse des doublons exacts terminée.")


In [ ]:
import json

thresholds_path = ARTIFACTS_DIR / "nn_thresholds.json"

print("Fichier présent :", thresholds_path.exists())

if thresholds_path.exists():
    with open(thresholds_path, "r", encoding="utf-8") as f:
        nn_thresholds = json.load(f)

    print("\nType :", type(nn_thresholds).__name__)
    print("Clés principales :", list(nn_thresholds.keys()))

    print("\nAperçu du contenu :")
    print(json.dumps(nn_thresholds, indent=2, ensure_ascii=False)[:5000])
else:
    print("Fichier nn_thresholds.json introuvable")

In [ ]:

import numpy as np
import pandas as pd

from sklearn.neighbors import NearestNeighbors

priority_classes = [
    "PortScan",
    "Bot",
    "DoS Slowhttptest"
]

results_nn = []
nearest_distances = {}

for class_name in priority_classes:

    print(f"\nAnalyse : {class_name}")

    # Données réelles du train, même classe
    real = train_df.loc[
        train_df["label"] == class_name,
        features
    ].to_numpy(dtype=np.float32)

    # Données synthétiques normalisées
    fake = np.asarray(
        generated_by_class[class_name],
        dtype=np.float32
    )

    if len(real) == 0 or len(fake) == 0:
        raise ValueError(
            f"Données manquantes pour {class_name}"
        )

    tau = float(nn_thresholds["tau_per_class"][class_name])

    # Recherche exacte avec la distance euclidienne
    nn = NearestNeighbors(
        n_neighbors=1,
        metric="euclidean",
        algorithm="brute",
        n_jobs=-1
    )

    nn.fit(real)

    # Recherche par petits lots
    distances_parts = []

    for start in range(0, len(fake), 256):
        batch = fake[start:start + 256]
        distances, _ = nn.kneighbors(batch)

        distances_parts.append(distances[:, 0])

    distances = np.concatenate(distances_parts)
    nearest_distances[class_name] = distances

    quasi_duplicates = int(np.sum(distances <= tau))
    rate = 100 * quasi_duplicates / len(distances)

    results_nn.append({
        "classe": class_name,
        "n_generated": len(fake),
        "tau": tau,
        "quasi_duplicates": quasi_duplicates,
        "quasi_duplicate_rate_pct": rate,
        "median_nn_distance": float(np.median(distances)),
        "min_nn_distance": float(np.min(distances))
    })

    print("Seuil :", tau)
    print("Quasi-doublons :", quasi_duplicates)
    print("Taux :", round(rate, 4), "%")

nn_results_df = pd.DataFrame(results_nn)

display(nn_results_df)

# Enregistrer les résultats
analysis_dir = OUTPUT_DIR / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)

nn_results_df.to_csv(
    analysis_dir / "quasi_duplicates_by_class.csv",
    index=False
)

print("\nAnalyse des quasi-doublons terminée.")


## 17. Comparaison des corrélations

Les matrices de corrélation du train réel et des données synthétiques sont comparées, classe prioritaire par classe prioritaire, sur les paires de caractéristiques définies.

In [ ]:
import numpy as np
import pandas as pd

priority_classes = [
    "PortScan",
    "Bot",
    "DoS Slowhttptest"
]

results_corr = []

for class_name in priority_classes:

    real = train_df.loc[
        train_df["label"] == class_name,
        features
    ]

    fake = pd.DataFrame(
        generated_by_class[class_name],
        columns=features
    )

    corr_real = real.corr().to_numpy()
    corr_fake = fake.corr().to_numpy()

    # Ignorer la diagonale et les corrélations non définies
    upper = np.triu(
        np.ones(corr_real.shape, dtype=bool),
        k=1
    )

    valid = (
        upper
        & np.isfinite(corr_real)
        & np.isfinite(corr_fake)
    )

    if not np.any(valid):
        raise ValueError(
            f"Aucune paire de corrélations valide pour {class_name}"
        )

    diff = np.abs(
        corr_real[valid] - corr_fake[valid]
    )

    results_corr.append({
        "classe": class_name,
        "n_pairs": int(valid.sum()),
        "mean_absolute_corr_diff": float(diff.mean()),
        "median_absolute_corr_diff": float(np.median(diff)),
        "max_absolute_corr_diff": float(diff.max())
    })

corr_results_df = pd.DataFrame(results_corr)

display(corr_results_df)

analysis_dir = OUTPUT_DIR / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)

corr_results_df.to_csv(
    analysis_dir / "correlation_comparison.csv",
    index=False
)

print("Analyse des corrélations terminée.")

### Diagnostic complémentaire des contraintes

Cette cellule inventorie les fonctions de contrôle disponibles dans le module de contraintes ; elle est conservée à sa position d'origine pour respecter le déroulé exécuté sur Colab.

In [ ]:
import inspect
import cvgen_data.constraints as constraints

print("Fonctions disponibles :")

for name, obj in inspect.getmembers(constraints):
    if inspect.isfunction(obj) and not name.startswith("_"):
        print(name, inspect.signature(obj))

### Visualisations complémentaires

Ces cellules complètent les analyses précédentes avec les courbes d'entraînement, les heatmaps 8×8, les histogrammes et les figures persistées sur Drive.

In [ ]:
print("Type de layout :", type(layout))
print("Transformation disponible :", hasattr(layout, "to_matrix"))

In [ ]:
metrics = pd.DataFrame(history)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, column in zip(axes, ['critic_loss','generator_loss','gradient_penalty']):
    ax.plot(metrics['epoch'], metrics[column]); ax.set_title(column); ax.set_xlabel('époque')
plt.tight_layout(); plt.show()

chosen_class = 'PortScan'
real_vector = train_df.loc[train_df.label == chosen_class, features].iloc[0].to_numpy(np.float32)[None]
real_matrix = layout.to_matrix(real_vector)[0, 0]
fake_matrix = priority_matrices[chosen_class][0, 0]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, matrix, title in zip(axes, [real_matrix, fake_matrix], ['Réel train','Synthétique brut']):
    sns.heatmap(matrix, vmin=-1, vmax=1, cmap='coolwarm', ax=ax); ax.set_title(f'{chosen_class} — {title}')
plt.tight_layout(); plt.show()

selected_features = ['flow_duration','flow_bytes_s','flow_packets_s']
fig, axes = plt.subplots(len(priority), len(selected_features), figsize=(15, 10))
for row, class_name in enumerate(priority):
    real = train_df[train_df.label == class_name]
    fake = generated_by_class[class_name]
    for col, feature in enumerate(selected_features):
        idx = features.index(feature); ax = axes[row, col]
        ax.hist(real[feature], bins=40, density=True, histtype='step', label='train réel')
        ax.hist(fake[:, idx], bins=40, density=True, histtype='step', label='synthétique')
        ax.set_title(f'{class_name} — {feature}'); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(len(priority), len(selected_features), figsize=(15, 10))
for row, class_name in enumerate(priority):
    tr = train_df[train_df.label == class_name]; va = val_df[val_df.label == class_name]
    for col, feature in enumerate(selected_features):
        ax = axes[row, col]
        ax.hist(tr[feature], bins=40, density=True, histtype='step', label='train')
        ax.hist(va[feature], bins=40, density=True, histtype='step', label='validation')
        ax.set_title(f'{class_name} — {feature}'); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
import matplotlib.pyplot as plt

metrics = pd.DataFrame(history)

best_epoch = 50

plt.figure(figsize=(9, 4))

plt.plot(
    metrics["epoch"],
    metrics["val_distribution_distance"],
    label="Distance de validation"
)

plt.axvline(
    best_epoch,
    color="red",
    linestyle="--",
    label="Meilleure époque : 50"
)

plt.xlabel("Époque")
plt.ylabel("Distance de validation")
plt.title("WGAN-GP — Évolution de la distance de validation")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
from pathlib import Path

print("OUTPUT_DIR :", OUTPUT_DIR)
print("OUTPUT_DIR existe :", OUTPUT_DIR.exists())

figures_dir = OUTPUT_DIR / "figures"

print("Dossier figures existe :", figures_dir.exists())

if figures_dir.exists():
    print("\nFichiers présents :")
    for f in figures_dir.iterdir():
        print(f.name)

In [ ]:
figures_dir = OUTPUT_DIR / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

print("Dossier créé :", figures_dir)
print("Existe :", figures_dir.exists())

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

# Dossiers
metrics_path = OUTPUT_DIR / "checkpoints" / "metrics.json"
figures_dir = OUTPUT_DIR / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

# Charger l'historique enregistré
with open(metrics_path, "r", encoding="utf-8") as f:
    history_saved = json.load(f)

metrics = pd.DataFrame(history_saved)

print("Nombre d'époques :", len(metrics))
print("Colonnes :", metrics.columns.tolist())

# Figure 1 : pertes et gradient penalty
columns = [
    "critic_loss",
    "generator_loss",
    "gradient_penalty"
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, column in zip(axes, columns):
    ax.plot(metrics["epoch"], metrics[column])
    ax.set_title(column)
    ax.set_xlabel("Époque")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(figures_dir / "01_training_losses.png", dpi=200)
plt.show()

# Figure 2 : distance de validation
best_idx = metrics["val_distribution_distance"].idxmin()
best_epoch = metrics.loc[best_idx, "epoch"]
best_distance = metrics.loc[best_idx, "val_distribution_distance"]

plt.figure(figsize=(9, 5))

plt.plot(
    metrics["epoch"],
    metrics["val_distribution_distance"],
    label="Distance de validation"
)

plt.axvline(
    best_epoch,
    color="red",
    linestyle="--",
    label=f"Meilleure époque : {best_epoch:.0f}"
)

plt.scatter(
    [best_epoch],
    [best_distance],
    color="red",
    zorder=5
)

plt.xlabel("Époque")
plt.ylabel("Distance de validation")
plt.title("WGAN-GP — Sélection du meilleur checkpoint")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(figures_dir / "02_validation_distance.png", dpi=200)
plt.show()

print("\nMeilleure époque :", best_epoch)
print("Meilleure distance :", best_distance)
print("Figures sauvegardées dans :", figures_dir)

## 18. Export des données synthétiques et des livrables

Les versions normalisée, inversée brute et post-traitée sont exportées séparément, avec le checkpoint sélectionné, la configuration, les artefacts et la provenance. Les Parquet et checkpoints restent sur Google Drive et ne doivent pas être ajoutés à Git.

In [ ]:
from cvgen_data.scaling import FlowScaler
export_dir = OUTPUT_DIR / 'generated'
export_dir.mkdir(parents=True, exist_ok=True)
scaler = FlowScaler.from_json(str(ARTIFACTS_DIR / 'scaler.json'))
with open(ARTIFACTS_DIR / 'feature_schema.json', encoding='utf-8') as handle:
    feature_schema = json.load(handle)
normalized_parts = []
for class_name, values in generated_by_class.items():
    part = pd.DataFrame(values, columns=features)
    part['label'] = class_name
    part['label_index'] = label_to_index[class_name]
    normalized_parts.append(part)
normalized = pd.concat(normalized_parts, ignore_index=True)
raw_inverse = scaler.inverse_transform(normalized[features], postprocess=False)
raw_inverse['label'] = normalized['label']; raw_inverse['label_index'] = normalized['label_index']
postprocessed = scaler.inverse_transform(normalized[features], postprocess=True, schema=feature_schema)
postprocessed['label'] = normalized['label']; postprocessed['label_index'] = normalized['label_index']
normalized.to_parquet(export_dir / 'synthetic_normalized_unprocessed.parquet', index=False)
raw_inverse.to_parquet(export_dir / 'synthetic_raw_inverse.parquet', index=False)
postprocessed.to_parquet(export_dir / 'synthetic_postprocessed.parquet', index=False)
shutil.copy2(selected_checkpoint, export_dir / 'selected_checkpoint.pt')
shutil.copy2(PROJECT_ROOT / 'cybervision_gan/configs/model_a.yaml', export_dir / 'model_a.yaml')
for name in ['label_encoding.json','matrix_layout.json','scaler.json','feature_schema.json']:
    shutil.copy2(ARTIFACTS_DIR / name, export_dir / name)
provenance = {
    'seed': SEED, 'checkpoint': str(selected_checkpoint), 'features': features,
    'label_encoding': label_to_index, 'uses_train': True, 'uses_validation': True,
    'uses_test': False, 'results_provisional_timestamp_audit': True,
}
(export_dir / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
print('Exports :', sorted(path.name for path in export_dir.iterdir()))

In [ ]:
from pathlib import Path

folders = {
    "Checkpoints": OUTPUT_DIR / "checkpoints",
    "Données générées": OUTPUT_DIR / "generated",
    "Analyses": OUTPUT_DIR / "analysis",
    "Graphiques": OUTPUT_DIR / "figures",
}

for category, folder in folders.items():
    print(f"\n=== {category} ===")

    if not folder.exists():
        print("DOSSIER ABSENT")
        continue

    for path in sorted(folder.iterdir()):
        if path.is_file():
            size_mb = path.stat().st_size / (1024 ** 2)
            print(f"{path.name} — {size_mb:.2f} Mo")

In [ ]:
import pandas as pd
import numpy as np

export_dir = OUTPUT_DIR / "generated"

filenames = [
    "synthetic_normalized_unprocessed.parquet",
    "synthetic_raw_inverse.parquet",
    "synthetic_postprocessed.parquet"
]

for filename in filenames:
    df = pd.read_parquet(export_dir / filename)

    assert all(feature in df.columns for feature in features)
    assert "label" in df.columns
    assert "label_index" in df.columns

    X = df[features].to_numpy(dtype=np.float64)

    assert np.isfinite(X).all(), f"NaN ou Inf : {filename}"
    assert set(df["label"].unique()) == set(label_to_index)

    print(
        filename,
        "| lignes :", len(df),
        "| colonnes :", len(df.columns),
        "| classes :", df["label"].nunique()
    )

## 19. Bilan technique et limites du modèle A

Le pipeline couvre l'entraînement conditionnel, la sélection sur validation, la génération des 11 classes, la projection 8×8, les contrôles de contraintes, les doublons et les corrélations, puis l'export traçable.

### Résultats expérimentaux rapportés

- 100 époques d'entraînement ; meilleur checkpoint à l'époque 50 ; distance de validation minimale : 0,071821.
- 23 000 exemples synthétiques générés sur 11 classes.
- Aucun doublon exact ou quasi-doublon détecté parmi les 15 000 exemples prioritaires selon les contrôles réalisés.
- Taux de lignes synthétiques violant au moins une contrainte fiable : 100 %, avant et après post-traitement.

Ces résultats proviennent du run Colab antérieur et n'ont pas été reproduits lors du nettoyage du notebook.

### Limites et travaux restants

- La plausibilité physique est insuffisante malgré le post-traitement.
- L'audit temporel des données doit être finalisé.
- L'utilité des données synthétiques pour une tâche de classification reste à démontrer.
- Les Parquet, checkpoints, figures et rapports CSV sont conservés sur Google Drive, hors du dépôt Git.